# PanNuke Dataset Analysis

Анализ датасета PanNuke для понимания распределений классов, размеров ядер, типов тканей и выявления проблем, которые могут влиять на качество модели.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('/home/corzent/caspian/thesis/datasets/pannuke')
CLASS_NAMES = ['Neoplastic', 'Inflammatory', 'Connective', 'Dead', 'Epithelial']
CLASS_COLORS = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

# Load all 3 folds (mmap)
folds = {}
for f in [1, 2, 3]:
    fold_dir = DATA_DIR / f'fold{f}'
    folds[f] = {
        'images': np.load(fold_dir / 'images.npy', mmap_mode='r'),
        'masks': np.load(fold_dir / 'masks.npy', mmap_mode='r'),
        'types': np.load(fold_dir / 'types.npy', mmap_mode='r'),
    }
    print(f'Fold {f}: {len(folds[f]["images"])} patches, images={folds[f]["images"].shape}, masks={folds[f]["masks"].shape}')

total = sum(len(folds[f]['images']) for f in folds)
print(f'\nTotal: {total} patches')

## 1. Распределение по фолдам и тканям

In [ ]:
# Tissue type distribution per fold
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, f in enumerate([1, 2, 3]):
    types = np.array(folds[f]['types'])
    counts = Counter(types)
    labels = [str(t) for t in sorted(counts.keys())]
    values = [counts[t] for t in sorted(counts.keys())]
    axes[i].barh(labels, values, color='steelblue')
    axes[i].set_title(f'Fold {f} ({len(types)} patches)')
    axes[i].set_xlabel('Patches')
plt.tight_layout()
plt.savefig(DATA_DIR.parent.parent / 'notebooks' / 'tissue_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Распределение классов ядер

In [ ]:
# Count nuclei per class across all folds
class_counts = defaultdict(int)  # class_idx -> total nuclei count
class_pixel_counts = defaultdict(int)  # class_idx -> total pixel count
nuclei_sizes = defaultdict(list)  # class_idx -> list of sizes (pixels per nucleus)
patches_with_class = defaultdict(int)  # class_idx -> number of patches containing this class
empty_patches = 0

for f in [1, 2, 3]:
    masks = folds[f]['masks']
    n_patches = len(masks)
    for idx in range(n_patches):
        m = np.array(masks[idx])  # (256, 256, 6)
        has_any = False
        for cls in range(5):
            cls_mask = m[:, :, cls]
            unique_ids = np.unique(cls_mask)
            unique_ids = unique_ids[unique_ids > 0]
            if len(unique_ids) > 0:
                has_any = True
                class_counts[cls] += len(unique_ids)
                patches_with_class[cls] += 1
                for uid in unique_ids:
                    size = (cls_mask == uid).sum()
                    nuclei_sizes[cls].append(size)
                    class_pixel_counts[cls] += size
        if not has_any:
            empty_patches += 1

print('=== Nuclei Statistics ===')
total_nuclei = sum(class_counts.values())
for cls in range(5):
    n = class_counts[cls]
    pct = 100 * n / total_nuclei if total_nuclei > 0 else 0
    avg_size = np.mean(nuclei_sizes[cls]) if nuclei_sizes[cls] else 0
    med_size = np.median(nuclei_sizes[cls]) if nuclei_sizes[cls] else 0
    print(f'{CLASS_NAMES[cls]:15s}: {n:6d} nuclei ({pct:5.1f}%), '
          f'in {patches_with_class[cls]:4d} patches, '
          f'avg size={avg_size:.0f}px, median={med_size:.0f}px')
print(f'\nTotal nuclei: {total_nuclei}')
print(f'Empty patches (no nuclei): {empty_patches}')

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart - nuclei counts
counts = [class_counts[c] for c in range(5)]
axes[0].bar(CLASS_NAMES, counts, color=CLASS_COLORS)
axes[0].set_ylabel('Number of nuclei')
axes[0].set_title('Class Distribution (nuclei count)')
for i, v in enumerate(counts):
    axes[0].text(i, v + 500, f'{v}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(counts, labels=CLASS_NAMES, colors=CLASS_COLORS, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution (%)')

plt.tight_layout()
plt.savefig(DATA_DIR.parent.parent / 'notebooks' / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Распределение размеров ядер по классам

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for cls in range(5):
    sizes = nuclei_sizes[cls]
    if sizes:
        axes[cls].hist(sizes, bins=50, color=CLASS_COLORS[cls], alpha=0.8, range=(0, 2000))
    axes[cls].set_title(f'{CLASS_NAMES[cls]}\n(n={len(sizes)}, med={np.median(sizes):.0f}px)')
    axes[cls].set_xlabel('Size (pixels)')
    axes[cls].axvline(np.median(sizes) if sizes else 0, color='black', linestyle='--', alpha=0.5)

axes[0].set_ylabel('Count')
plt.suptitle('Nucleus Size Distribution by Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR.parent.parent / 'notebooks' / 'size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Плотность ядер на патч

In [ ]:
# Nuclei density per patch
nuclei_per_patch = []
class_per_patch = defaultdict(list)  # cls -> [count_per_patch]

for f in [1, 2, 3]:
    masks = folds[f]['masks']
    for idx in range(len(masks)):
        m = np.array(masks[idx])
        total_in_patch = 0
        for cls in range(5):
            n = len(np.unique(m[:, :, cls])) - 1  # exclude 0
            class_per_patch[cls].append(n)
            total_in_patch += n
        nuclei_per_patch.append(total_in_patch)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(nuclei_per_patch, bins=60, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Nuclei per patch')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Nuclei Density per Patch\n(mean={np.mean(nuclei_per_patch):.1f}, median={np.median(nuclei_per_patch):.0f})')
axes[0].axvline(np.mean(nuclei_per_patch), color='red', linestyle='--', label=f'mean={np.mean(nuclei_per_patch):.1f}')
axes[0].legend()

# Box plot per class
data = [class_per_patch[c] for c in range(5)]
bp = axes[1].boxplot(data, labels=CLASS_NAMES, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], CLASS_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_ylabel('Nuclei per patch')
axes[1].set_title('Nuclei Count per Patch by Class')

plt.tight_layout()
plt.savefig(DATA_DIR.parent.parent / 'notebooks' / 'density_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Примеры патчей

In [ ]:
def visualize_patch(image, masks, ax_img, ax_mask, title=''):
    """Show image and colored instance mask overlay."""
    ax_img.imshow(image)
    ax_img.set_title(title)
    ax_img.axis('off')
    
    # Create colored overlay
    overlay = np.zeros((*image.shape[:2], 4), dtype=np.float32)
    cls_colors_rgba = [
        [0.89, 0.10, 0.11, 0.5],  # red - neoplastic
        [0.22, 0.49, 0.72, 0.5],  # blue - inflammatory
        [0.30, 0.69, 0.29, 0.5],  # green - connective
        [0.60, 0.31, 0.64, 0.5],  # purple - dead
        [1.00, 0.50, 0.00, 0.5],  # orange - epithelial
    ]
    for cls in range(5):
        cls_mask = masks[:, :, cls] > 0
        for c in range(4):
            overlay[:, :, c][cls_mask] = cls_colors_rgba[cls][c]
    
    ax_mask.imshow(image)
    ax_mask.imshow(overlay)
    ax_mask.set_title('Annotations')
    ax_mask.axis('off')

# Show 8 random patches from different tissue types
np.random.seed(42)
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

fold1_imgs = folds[1]['images']
fold1_masks = folds[1]['masks']
fold1_types = np.array(folds[1]['types'])

# Select patches from different tissue types
unique_tissues = np.unique(fold1_types)
selected = []
for t in np.random.choice(unique_tissues, min(4, len(unique_tissues)), replace=False):
    indices = np.where(fold1_types == t)[0]
    selected.append(np.random.choice(indices))

for idx in np.random.choice(len(fold1_imgs), 4, replace=False):
    if idx not in selected:
        selected.append(idx)
selected = selected[:4]

for row, idx in enumerate(selected):
    img = np.array(fold1_imgs[idx])
    msk = np.array(fold1_masks[idx])
    tissue = str(fold1_types[idx])
    
    n_nuclei = sum(len(np.unique(msk[:, :, c])) - 1 for c in range(5))
    visualize_patch(img, msk, axes[row, 0], axes[row, 1], f'{tissue} (n={n_nuclei})')
    
    for c in range(2):
        cls_mask = msk[:, :, c] > 0
        axes[row, c+2].imshow(cls_mask, cmap='gray')
        n_cls = len(np.unique(msk[:, :, c])) - 1
        axes[row, c+2].set_title(f'{CLASS_NAMES[c]} (n={n_cls})')
        axes[row, c+2].axis('off')

handles = [mpatches.Patch(color=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=12)
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig(DATA_DIR.parent.parent / 'notebooks' / 'sample_patches.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Дисбаланс классов и пиксельное покрытие

In [ ]:
# Pixel-level class imbalance
total_pixels = total * 256 * 256
total_fg_pixels = sum(class_pixel_counts.values())
bg_pixels = total_pixels - total_fg_pixels

print('=== Pixel-level Statistics ===')
print(f'Total pixels: {total_pixels:,}')
print(f'Background pixels: {bg_pixels:,} ({100*bg_pixels/total_pixels:.1f}%)')
print(f'Foreground pixels: {total_fg_pixels:,} ({100*total_fg_pixels/total_pixels:.1f}%)')
print()

for cls in range(5):
    px = class_pixel_counts[cls]
    pct_of_fg = 100 * px / total_fg_pixels if total_fg_pixels > 0 else 0
    pct_of_total = 100 * px / total_pixels
    print(f'{CLASS_NAMES[cls]:15s}: {px:10,} pixels ({pct_of_fg:5.1f}% of FG, {pct_of_total:5.2f}% of total)')

# Compute class weights (inverse frequency)
print('\n=== Suggested Class Weights (inverse frequency) ===')
freqs = np.array([class_pixel_counts[c] for c in range(5)], dtype=np.float64)
freqs = freqs / freqs.sum()
weights = 1.0 / (freqs + 1e-8)
weights = weights / weights.min()  # normalize so min weight = 1
for cls in range(5):
    print(f'{CLASS_NAMES[cls]:15s}: weight = {weights[cls]:.2f}')

In [ ]:
# Visualize pixel imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FG vs BG
axes[0].pie([bg_pixels, total_fg_pixels], 
            labels=['Background', 'Foreground'], 
            colors=['#cccccc', '#2196F3'],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Background vs Foreground Pixels')

# Per-class pixel share (foreground only)
fg_counts = [class_pixel_counts[c] for c in range(5)]
axes[1].bar(CLASS_NAMES, fg_counts, color=CLASS_COLORS)
axes[1].set_ylabel('Pixel count')
axes[1].set_title('Pixel Coverage per Class (foreground only)')
for i, v in enumerate(fg_counts):
    axes[1].text(i, v + 1000, f'{100*v/total_fg_pixels:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(DATA_DIR.parent.parent / 'notebooks' / 'pixel_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Анализ HV-map и watershed качества GT

In [ ]:
from scipy.ndimage import center_of_mass

def generate_hv_map(instance_map):
    h, w = instance_map.shape
    hv = np.zeros((2, h, w), dtype=np.float32)
    for inst_id in np.unique(instance_map):
        if inst_id == 0:
            continue
        mask = instance_map == inst_id
        ys, xs = np.where(mask)
        if len(ys) == 0:
            continue
        cy, cx = center_of_mass(mask)
        x_dist = xs - cx
        y_dist = ys - cy
        x_max = max(np.max(np.abs(x_dist)), 1)
        y_max = max(np.max(np.abs(y_dist)), 1)
        hv[0, ys, xs] = x_dist / x_max
        hv[1, ys, xs] = y_dist / y_max
    return hv

# Show HV maps for a sample patch with many nuclei
# Find a dense patch
dense_idx = np.argmax(nuclei_per_patch[:len(folds[1]['images'])])
img = np.array(folds[1]['images'][dense_idx])
msk = np.array(folds[1]['masks'][dense_idx])

# Build instance map
instance_map = np.zeros((256, 256), dtype=np.int32)
inst_id = 0
for cls in range(5):
    for uid in np.unique(msk[:, :, cls]):
        if uid == 0:
            continue
        inst_id += 1
        instance_map[msk[:, :, cls] == uid] = inst_id

hv = generate_hv_map(instance_map)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img)
axes[0].set_title(f'Image (n={inst_id} nuclei)')
axes[0].axis('off')

axes[1].imshow(instance_map > 0, cmap='gray')
axes[1].set_title('Binary mask')
axes[1].axis('off')

axes[2].imshow(hv[0], cmap='RdBu_r', vmin=-1, vmax=1)
axes[2].set_title('Horizontal distance map')
axes[2].axis('off')

axes[3].imshow(hv[1], cmap='RdBu_r', vmin=-1, vmax=1)
axes[3].set_title('Vertical distance map')
axes[3].axis('off')

plt.tight_layout()
plt.savefig(DATA_DIR.parent.parent / 'notebooks' / 'hv_maps_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Анализ ошибок текущей модели

In [ ]:
import sys
sys.path.insert(0, '/home/corzent/caspian/thesis')

import torch
from cellvit_distill.data.pannuke import PanNukeDataset, get_val_transform
from cellvit_distill.models.student import build_student
from cellvit_distill.utils.postprocess import post_process_predictions
from cellvit_distill.utils.metrics import panoptic_quality

# Load best baseline model
import yaml, json
from pathlib import Path

run_dir = Path('/home/corzent/caspian/thesis/cellvit_distill/runs/baseline_convnext_tiny_20260405_101901')
with open(run_dir / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_student(cfg).to(device)
ckpt = torch.load(run_dir / 'best_model.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded baseline model from epoch {ckpt["epoch"]}')

# Load test set
ds = PanNukeDataset(
    cfg['data']['data_dir'], folds=[3],
    transform=get_val_transform()
)
print(f'Test set: {len(ds)} patches')

In [ ]:
# Evaluate and find worst/best patches
patch_scores = []  # (idx, pq, n_gt, n_pred)

with torch.no_grad():
    for idx in range(min(500, len(ds))):  # sample 500 patches
        sample = ds[idx]
        img = sample['image'].unsqueeze(0).to(device)
        
        with torch.amp.autocast('cuda'):
            out = model(img)
        
        binary_pred = torch.softmax(out['binary'].float(), dim=1)[0, 1].cpu().numpy()
        hv_pred = out['hv_map'].float()[0].cpu().numpy()
        type_pred = torch.softmax(out['type_map'].float(), dim=1)[0].cpu().numpy()
        
        pred_inst, pred_type = post_process_predictions(binary_pred, hv_pred, type_pred)
        gt_inst = sample['instance_map'].numpy()
        
        result = panoptic_quality(pred_inst, gt_inst)
        n_gt = len(np.unique(gt_inst)) - 1
        n_pred = len(np.unique(pred_inst)) - 1
        
        patch_scores.append((idx, result['PQ'], n_gt, n_pred, result['TP'], result['FP'], result['FN']))
        
        if idx % 100 == 0:
            print(f'{idx}/{min(500, len(ds))}...')

patch_scores.sort(key=lambda x: x[1])  # sort by PQ ascending
pqs = [s[1] for s in patch_scores]
print(f'\nPQ statistics: mean={np.mean(pqs):.3f}, median={np.median(pqs):.3f}, '
      f'min={np.min(pqs):.3f}, max={np.max(pqs):.3f}')

In [ ]:
# Distribution of per-patch PQ
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(pqs, bins=40, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Panoptic Quality')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Per-Patch PQ Distribution (n={len(pqs)})')
axes[0].axvline(np.mean(pqs), color='red', linestyle='--', label=f'mean={np.mean(pqs):.3f}')
axes[0].legend()

# FP/FN analysis
fps = [s[5] for s in patch_scores]
fns = [s[6] for s in patch_scores]
axes[1].scatter(fns, fps, alpha=0.3, c=pqs, cmap='RdYlGn', s=15)
axes[1].set_xlabel('False Negatives (missed nuclei)')
axes[1].set_ylabel('False Positives (spurious nuclei)')
axes[1].set_title('FP vs FN per patch')
axes[1].plot([0, max(fns)], [0, max(fns)], 'k--', alpha=0.3)

# GT count vs prediction count
n_gts = [s[2] for s in patch_scores]
n_preds = [s[3] for s in patch_scores]
axes[2].scatter(n_gts, n_preds, alpha=0.3, c=pqs, cmap='RdYlGn', s=15)
axes[2].set_xlabel('GT nuclei count')
axes[2].set_ylabel('Predicted nuclei count')
axes[2].set_title('Predicted vs GT nuclei count')
axes[2].plot([0, max(n_gts)], [0, max(n_gts)], 'k--', alpha=0.5)

plt.tight_layout()
plt.savefig(DATA_DIR.parent.parent / 'notebooks' / 'error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize worst and best predictions
def show_prediction(idx, ax_row, title_prefix=''):
    sample = ds[idx]
    img_tensor = sample['image']
    # Denormalize
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_vis = (img_tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    
    with torch.no_grad(), torch.amp.autocast('cuda'):
        out = model(img_tensor.unsqueeze(0).to(device))
    
    binary_pred = torch.softmax(out['binary'].float(), dim=1)[0, 1].cpu().numpy()
    hv_pred = out['hv_map'].float()[0].cpu().numpy()
    type_pred = torch.softmax(out['type_map'].float(), dim=1)[0].cpu().numpy()
    pred_inst, pred_type = post_process_predictions(binary_pred, hv_pred, type_pred)
    gt_inst = sample['instance_map'].numpy()
    
    result = panoptic_quality(pred_inst, gt_inst)
    
    ax_row[0].imshow(img_vis)
    ax_row[0].set_title(f'{title_prefix}Image')
    ax_row[0].axis('off')
    
    ax_row[1].imshow(gt_inst > 0, cmap='gray')
    ax_row[1].set_title(f'GT ({len(np.unique(gt_inst))-1} nuclei)')
    ax_row[1].axis('off')
    
    ax_row[2].imshow(binary_pred, cmap='hot', vmin=0, vmax=1)
    ax_row[2].set_title('Binary prediction')
    ax_row[2].axis('off')
    
    ax_row[3].imshow(pred_inst > 0, cmap='gray')
    ax_row[3].set_title(f'Pred ({len(np.unique(pred_inst))-1} nuclei)\nPQ={result["PQ"]:.3f}')
    ax_row[3].axis('off')

fig, axes = plt.subplots(4, 4, figsize=(20, 20))

# 2 worst
worst = [s for s in patch_scores if s[2] > 5][:2]  # worst with >5 GT nuclei
for i, (idx, pq, *_) in enumerate(worst):
    show_prediction(idx, axes[i], f'WORST (PQ={pq:.3f}) ')

# 2 best
best = [s for s in reversed(patch_scores) if s[2] > 5][:2]
for i, (idx, pq, *_) in enumerate(best):
    show_prediction(idx, axes[i+2], f'BEST (PQ={pq:.3f}) ')

plt.tight_layout()
plt.savefig(DATA_DIR.parent.parent / 'notebooks' / 'worst_best_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Выводы и направления улучшения

На основе анализа:

1. **Класс Dead** — крайне редкий, модель не может его выучить → нужна oversampling стратегия или focal loss
2. **Класс Epithelial** — низкий PQ несмотря на достаточное количество → возможно, сложная морфология, нужно больше capacity
3. **Дисбаланс BG/FG** — ~95% пикселей это фон → weighted loss или hard example mining
4. **Переобучение после epoch 20** → более агрессивная регуляризация (dropout, weight decay, augmentation)
5. **Температура дистилляции** — T=4 недостаточно для сглаживания уверенных логитов учителя → T=10-20